# Eksperimen Preprocessing — Poker Hand Dataset
**Nama:** ASEP HARYANA SAPUTRA  
**Dataset:** Poker Hand (UCI ML Repository)  
**Deskripsi:** Dataset berisi informasi tangan poker yang terdiri dari 5 kartu. Setiap kartu dideskripsikan dengan suit (jenis kartu) dan rank (nilai kartu). Target: klasifikasi 10 kelas tangan poker.

---

## 1️⃣ Import Library

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from collections import Counter

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded successfully.')

## 2️⃣ Data Loading

In [ ]:
# Column definitions berdasarkan poker-hand.names
COLUMNS = [
    'S1', 'C1',  # Suit & Rank card 1
    'S2', 'C2',  # Suit & Rank card 2
    'S3', 'C3',  # Suit & Rank card 3
    'S4', 'C4',  # Suit & Rank card 4
    'S5', 'C5',  # Suit & Rank card 5
    'CLASS'      # Target: Poker Hand (0-9)
]

CLASS_NAMES = {
    0: 'Nothing',
    1: 'One Pair',
    2: 'Two Pairs',
    3: 'Three of a Kind',
    4: 'Straight',
    5: 'Flush',
    6: 'Full House',
    7: 'Four of a Kind',
    8: 'Straight Flush',
    9: 'Royal Flush'
}

RAW_DIR = '../pokerhand_raw'

# Load training set
df_train = pd.read_csv(
    os.path.join(RAW_DIR, 'poker-hand-training-true.data'),
    header=None,
    names=COLUMNS
)

# Sample 100k rows dari testing set untuk validasi EDA (full set = 1M rows)
df_test_sample = pd.read_csv(
    os.path.join(RAW_DIR, 'poker-hand-testing.data'),
    header=None,
    names=COLUMNS,
    nrows=100_000
)

print(f'Training shape: {df_train.shape}')
print(f'Test sample shape: {df_test_sample.shape}')

In [ ]:
# Head
df_train.head(10)

In [ ]:
# Info
df_train.info()

In [ ]:
# Descriptive statistics
df_train.describe()

In [ ]:
# ============================
# Cek Missing Values
# ============================
missing = df_train.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing: {missing.sum()}')

In [ ]:
# ============================
# Cek Duplicate
# ============================
n_dup = df_train.duplicated().sum()
print(f'Duplicate rows: {n_dup}')
print(f'Percentage: {n_dup / len(df_train) * 100:.2f}%')

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# ============================
# 3.1 Class Distribution
# ============================
class_counts = df_train['CLASS'].value_counts().sort_index()
class_labels = [f'{k}: {CLASS_NAMES[k]}' for k in class_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].bar(class_counts.index, class_counts.values, color=sns.color_palette('viridis', 10))
axes[0].set_xlabel('Poker Hand Class')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution (Linear Scale)')
axes[0].set_xticks(range(10))

# Log scale — reveals rare classes
axes[1].bar(class_counts.index, class_counts.values, color=sns.color_palette('viridis', 10))
axes[1].set_yscale('log')
axes[1].set_xlabel('Poker Hand Class')
axes[1].set_ylabel('Count (log scale)')
axes[1].set_title('Class Distribution (Log Scale)')
axes[1].set_xticks(range(10))

plt.suptitle('Target Variable (CLASS) Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nClass counts:')
for k, v in class_counts.items():
    print(f'  Class {k} ({CLASS_NAMES[k]}): {v} ({v/len(df_train)*100:.2f}%)')

In [ ]:
# ============================
# 3.2 Distribusi Fitur Numerik
# ============================
feature_cols = [c for c in COLUMNS if c != 'CLASS']
suit_cols = ['S1', 'S2', 'S3', 'S4', 'S5']
rank_cols = ['C1', 'C2', 'C3', 'C4', 'C5']

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(df_train[col], bins=df_train[col].nunique(), edgecolor='white', 
                 color='steelblue' if col.startswith('C') else 'coral')
    axes[i].set_title(f'Distribution of {col}', fontsize=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions (Blue=Rank, Coral=Suit)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# 3.3 Analisis Fitur Kategorikal (Suit)
# Suit values: 1=Hearts, 2=Spades, 3=Diamonds, 4=Clubs
# ============================
suit_map = {1: 'Hearts ♥', 2: 'Spades ♠', 3: 'Diamonds ♦', 4: 'Clubs ♣'}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, col in enumerate(suit_cols):
    counts = df_train[col].map(suit_map).value_counts()
    axes[i].bar(counts.index, counts.values, color=['#e74c3c', '#2c3e50', '#3498db', '#27ae60'])
    axes[i].set_title(f'{col} Suit Frequency')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylabel('Count')

plt.suptitle('Suit Distribution per Card Position', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# 3.4 Heatmap Korelasi
# ============================
plt.figure(figsize=(12, 9))
corr = df_train.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    vmin=-1, vmax=1
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# 3.5 Boxplot Rank per Class
# ============================
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for i, col in enumerate(rank_cols):
    # Sample untuk kecepatan plotting
    sample = df_train.sample(min(5000, len(df_train)), random_state=42)
    class_order = sorted(df_train['CLASS'].unique())
    sample['CLASS_NAME'] = sample['CLASS'].map(lambda x: f'C{x}')
    axes[i].boxplot(
        [df_train.loc[df_train['CLASS'] == c, col].values for c in class_order],
        labels=[f'C{c}' for c in class_order]
    )
    axes[i].set_title(f'{col} by Class')
    axes[i].set_xlabel('Poker Hand Class')
    axes[i].set_ylabel('Rank Value')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Card Rank Distribution per Poker Hand Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# 3.6 Outlier Analysis
# ============================
# Semua fitur punya range terbatas (Suit: 1-4, Rank: 1-13)
# Outlier secara statistik tidak possible di luar range ini.
# Namun kita cek IQR untuk distribusi rank.
print('IQR-based outlier check untuk Rank columns:')
for col in rank_cols:
    Q1 = df_train[col].quantile(0.25)
    Q3 = df_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_train[(df_train[col] < lower) | (df_train[col] > upper)]
    print(f'  {col}: Q1={Q1}, Q3={Q3}, IQR={IQR:.2f}, '
          f'bounds=[{lower:.2f}, {upper:.2f}], outliers={len(outliers)}')

In [ ]:
# ============================
# 3.7 Feature Engineering Insight
# Derived features yang relevan untuk poker
# ============================
def compute_derived_features(df):
    d = df.copy()
    # Apakah semua suit sama? (indikasi Flush)
    d['all_same_suit'] = (d[['S1','S2','S3','S4','S5']].nunique(axis=1) == 1).astype(int)
    # Jumlah suit unik
    d['n_unique_suits'] = d[['S1','S2','S3','S4','S5']].nunique(axis=1)
    # Jumlah rank unik
    d['n_unique_ranks'] = d[['C1','C2','C3','C4','C5']].nunique(axis=1)
    # Max rank - Min rank (rentang rank)
    d['rank_range'] = d[['C1','C2','C3','C4','C5']].max(axis=1) - d[['C1','C2','C3','C4','C5']].min(axis=1)
    return d

df_derived = compute_derived_features(df_train)
derived_cols = ['all_same_suit', 'n_unique_suits', 'n_unique_ranks', 'rank_range']

# Korelasi derived features dengan CLASS
print('Korelasi derived features dengan CLASS:')
print(df_derived[derived_cols + ['CLASS']].corr()['CLASS'].sort_values(ascending=False))

## 4️⃣ Data Preprocessing

In [ ]:
# ============================
# 4.1 Handle Missing Values
# ============================
# Berdasarkan EDA: dataset ini TIDAK memiliki missing values.
# Namun kita tetap validasi dan log.
assert df_train.isnull().sum().sum() == 0, 'Unexpected missing values found!'
print('✅ No missing values — no imputation needed.')

In [ ]:
# ============================
# 4.2 Handle Duplicates
# ============================
n_before = len(df_train)
df_clean = df_train.drop_duplicates().copy()
n_after = len(df_clean)
print(f'Rows before dedup: {n_before}')
print(f'Rows after dedup : {n_after}')
print(f'Removed          : {n_before - n_after} duplicates')

In [ ]:
# ============================
# 4.3 Feature Engineering
# ============================
df_feat = compute_derived_features(df_clean)
print('Dataset shape setelah feature engineering:', df_feat.shape)
df_feat[derived_cols + ['CLASS']].head()

In [ ]:
# ============================
# 4.4 Encoding Fitur Kategorikal (Suit)
# Suit bukan ordinal sebenarnya (Hearts ≠ < Spades),
# oleh karena itu kita One-Hot encode.
# ============================

# Pisahkan fitur dan target
X = df_feat.drop(columns=['CLASS'])
y = df_feat['CLASS']

# Definisi kolom berdasarkan jenis
suit_cols   = ['S1', 'S2', 'S3', 'S4', 'S5']
rank_cols   = ['C1', 'C2', 'C3', 'C4', 'C5']
derived_cols_list = ['all_same_suit', 'n_unique_suits', 'n_unique_ranks', 'rank_range']

# ColumnTransformer: OHE untuk suit, Standard Scaler untuk rank (dan derived numerics)
preprocessor = ColumnTransformer(
    transformers=[
        ('suit_ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), suit_cols),
        ('rank_scale', StandardScaler(), rank_cols + derived_cols_list),
    ],
    remainder='drop'
)

print('Preprocessor configured successfully.')
print(f'Input features: {X.shape[1]}')

In [ ]:
# ============================
# 4.5 Train-Test Split
# 80/20 split dengan stratification untuk menjaga distribusi kelas
# ============================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train_raw.shape}')
print(f'X_test shape : {X_test_raw.shape}')
print(f'y_train distribution:')
print(y_train.value_counts().sort_index())

In [ ]:
# ============================
# 4.6 Fit & Transform
# ============================
X_train_enc = preprocessor.fit_transform(X_train_raw)
X_test_enc  = preprocessor.transform(X_test_raw)

# Buat nama kolom output
ohe_feature_names = preprocessor.named_transformers_['suit_ohe'].get_feature_names_out(suit_cols).tolist()
scale_feature_names = rank_cols + derived_cols_list
all_feature_names = ohe_feature_names + scale_feature_names

print(f'X_train_enc shape: {X_train_enc.shape}')
print(f'X_test_enc shape : {X_test_enc.shape}')
print(f'Feature names ({len(all_feature_names)}): {all_feature_names}')

In [ ]:
# ============================
# 4.7 SMOTE — Handle Class Imbalance
# SMOTE hanya diaplikasikan pada training set!
# ============================
print('Class distribution BEFORE SMOTE:')
print(Counter(y_train))

smote = SMOTE(random_state=42, k_neighbors=3)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_enc, y_train)

print('\nClass distribution AFTER SMOTE:')
print(Counter(y_train_balanced))
print(f'\nShape after SMOTE: {X_train_balanced.shape}')

In [ ]:
# Visualisasi sebelum vs sesudah SMOTE
before = Counter(y_train)
after  = Counter(y_train_balanced)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = sorted(before.keys())
axes[0].bar(classes, [before[c] for c in classes], color=sns.color_palette('viridis', 10))
axes[0].set_title('Before SMOTE')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

axes[1].bar(classes, [after[c] for c in classes], color=sns.color_palette('plasma', 10))
axes[1].set_title('After SMOTE (Balanced)')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')

plt.suptitle('Class Imbalance Handling via SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# 4.8 Simpan Hasil Preprocessing
# ============================
OUT_DIR = 'pokerhand_preprocessing'
os.makedirs(OUT_DIR, exist_ok=True)

# Convert ke DataFrame dengan nama kolom
df_X_train = pd.DataFrame(X_train_balanced, columns=all_feature_names)
df_X_test  = pd.DataFrame(X_test_enc,  columns=all_feature_names)
df_y_train = pd.DataFrame(y_train_balanced, columns=['CLASS'])
df_y_test  = pd.DataFrame(y_test.values, columns=['CLASS'])

df_X_train.to_csv(os.path.join(OUT_DIR, 'X_train.csv'), index=False)
df_X_test.to_csv(os.path.join(OUT_DIR, 'X_test.csv'),  index=False)
df_y_train.to_csv(os.path.join(OUT_DIR, 'y_train.csv'), index=False)
df_y_test.to_csv(os.path.join(OUT_DIR, 'y_test.csv'),  index=False)

print('✅ Preprocessed files saved to:', OUT_DIR)
for fname in os.listdir(OUT_DIR):
    fpath = os.path.join(OUT_DIR, fname)
    size  = os.path.getsize(fpath) / 1024
    print(f'  {fname}: {size:.1f} KB')

In [ ]:
# Simpan juga preprocessor object untuk automasi script
import joblib
joblib.dump(preprocessor, os.path.join(OUT_DIR, 'preprocessor.joblib'))
print('✅ Preprocessor saved as preprocessor.joblib')

In [ ]:
# Final Summary
print('='*60)
print('PREPROCESSING SUMMARY')
print('='*60)
print(f'Raw training samples  : {len(df_train)}')
print(f'After dedup           : {len(df_clean)}')
print(f'After feature eng.    : {X.shape[1]} features (was 10)')
print(f'After encoding        : {X_train_enc.shape[1]} encoded features')
print(f'X_train (after SMOTE) : {X_train_balanced.shape}')
print(f'X_test                : {X_test_enc.shape}')
print(f'Classes balanced      : {len(set(y_train_balanced))}')
print('='*60)